In [ ]:
# c# Check GPU
# !nvidia-smi

In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/DataSet/GreenLens/RCLD"

classes = sorted(os.listdir(DATASET_PATH))
print("Number of classes:", len(classes))
print(classes)

for c in classes:
    print(c, "->", len(os.listdir(os.path.join(DATASET_PATH, c))), "images") #c folder


Number of classes: 11
['Bacterial Leaf Blight', 'Blast', 'Brown Spot', 'Healthy Rice Leaf', 'Leaf scald', 'Narrow Brown Leaf Spot', 'Sheath Blight', 'Tungro virus', 'grassy stunt virus', 'ragged stunt virus', 'yellow mottle1 virus']
Bacterial Leaf Blight -> 1824 images
Blast -> 3765 images
Brown Spot -> 2192 images
Healthy Rice Leaf -> 1738 images
Leaf scald -> 1960 images
Narrow Brown Leaf Spot -> 1004 images
Sheath Blight -> 2311 images
Tungro virus -> 1458 images
grassy stunt virus -> 150 images
ragged stunt virus -> 150 images
yellow mottle1 virus -> 50 images


In [ ]:
class_to_idx = {cls:i for i,cls in enumerate(classes)}
print(class_to_idx)


{'Bacterial Leaf Blight': 0, 'Blast': 1, 'Brown Spot': 2, 'Healthy Rice Leaf': 3, 'Leaf scald': 4, 'Narrow Brown Leaf Spot': 5, 'Sheath Blight': 6, 'Tungro virus': 7, 'grassy stunt virus': 8, 'ragged stunt virus': 9, 'yellow mottle1 virus': 10}


In [ ]:
import os

counts = {}
for cls in classes:
    counts[cls] = len(os.listdir(os.path.join(DATASET_PATH, cls)))

print(counts)


{'Bacterial Leaf Blight': 1824, 'Blast': 3765, 'Brown Spot': 2192, 'Healthy Rice Leaf': 1738, 'Leaf scald': 1960, 'Narrow Brown Leaf Spot': 1004, 'Sheath Blight': 2311, 'Tungro virus': 1458, 'grassy stunt virus': 150, 'ragged stunt virus': 150, 'yellow mottle1 virus': 50}


In [ ]:
import torch

class_weights = []
for cls in classes:
    class_weights.append(1.0 / counts[cls])

print(class_weights)


#we are giving more important for rare diseases. In this dataset we gives more importants to 'grassy stunt virus': 150, 'ragged stunt virus': 150, 'yellow mottle1 virus': 50 deseases.

[0.0005482456140350877, 0.0002656042496679947, 0.0004562043795620438, 0.0005753739930955121, 0.0005102040816326531, 0.00099601593625498, 0.00043271311120726956, 0.0006858710562414266, 0.006666666666666667, 0.006666666666666667, 0.02]


In [ ]:
# Your model must work for farmer photos
# But farmers take photos:

# tilted 📱

# dark 🌑 / bright ☀️

# blurred 😵

# half leaf 🍃

# finger covering camera 👆

# So we purposely damage the training images
# → so the model becomes strong.

# That’s all this code is doing.

# One sentence summary

# We intentionally make training images messy so the model works on real farmer photos.

#AND

# Wer'e hiding part of the leaf which actually HELPS

# In real life the farmer image is rarely perfect:

# leaf folded

# insect eaten

# shadow

# water drops

# finger covering camera

# half leaf only visible

# If model only trains on perfect images → it memorizes exact shapes.

# Then real photo comes → model fails ❌

# What CoarseDropout teaches

# Instead of learning:

# “Brown spot is at this exact pixel location”

# The model learns:

# “Brown spot is a TEXTURE PATTERN across the leaf”

# So even if 30% is hidden → it still recognizes disease.


import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 224

train_transform = A.Compose([
    A.Resize(256,256),
    A.RandomCrop(IMG_SIZE, IMG_SIZE),

    A.HorizontalFlip(p=0.5), #50% of images will be flipped left ↔ right
    A.VerticalFlip(p=0.3), #30% images upside-down
    A.Rotate(limit=25, p=0.5), #So we are showing the crop images in different angles for each diseases or classes, this is because model should identify the diseases even if farmer took photo in natural angles instead of took the crop photo in stright. range 25 to -25 "tilt"

    A.RandomBrightnessContrast(p=0.5), #50% of images will have lighting changed. Without this → model thinks color shade = disease ❌, Changes lighting intensity, brighter / darker, ➡️ Simulates sunlight, shadow, cloudy weather, Light problem
    A.HueSaturationValue(p=0.3), #30% of images will have color tone modified. Changes color tone, green → yellowish / bluish, Simulates different phone cameras & color balance, color problem
    A.GaussianBlur(p=0.2), #It makes the image slightly blurry, Out of 10 training images: 2 images → blurred & 8 images → normal
    A.CoarseDropout(num_holes_range=(1,8),
                    hole_height_range=(8,32),
                    hole_width_range=(8,32),
                    p=0.3), #1–8 = count of blocks 8–32 = size of blocks, Minimum → 1 block hidden, Maximum → 8 blocks hidden, Small block → 8 pixels tall, Big block → 32 pixels tall, Narrow → 8 pixels, Wide → 32 pixels

    A.Normalize(), #Images normally have pixel values: 0 to 255, Example pixel (RGB):[120, 200, 45] Neural networks don’t learn well with large values.So we shrink and center them. After normalize (approx):, -1 to 1 Example becomes: [-0.1, 0.5, -0.6]. A.Normalize() uses ImageNet statistics by default: new_pixel=std(pixel/255)−mean​/std
    ToTensorV2(), #PyTorch CNN cannot read images like this: Height × Width × Channels (224, 224, 3). It needs: Channels × Height × Width  (3, 224, 224) SO It converts: (224,224,3) → (3,224,224)
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
]) #Make every image same size (224×224) CNN needs fixed input size. No cropping, no flipping, no rotation ❌ We don’t want to modify the test image.we only distrub train image not test image, ToTensorv2 (H,W,C)→(C,H,W), A.Normalize() uses ImageNet statistics by default: new_pixel=std(pixel/255)−mean​/std



In [ ]:
# Every step the model says:

# “Give me next image”

# This code does:

# 1️⃣ Find the image path
# 2️⃣ Open the image
# 3️⃣ Fix if image is broken
# 4️⃣ Apply augmentation (flip, rotate, hide parts etc.)
# 5️⃣ Convert to tensor
# 6️⃣ Return label number

# Then CNN learns from it.

import cv2
import torch
from torch.utils.data import Dataset
import numpy as np

class RiceDiseaseDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        for cls in classes:
            class_path = os.path.join(root_dir, cls)
            label = class_to_idx[cls]

            for img in os.listdir(class_path):
                self.samples.append((os.path.join(class_path, img), label)) #Here all image path and label is added
#                 Index           Path             Label(Class)
                  # 0       BacterialLeaf/1.jpg      0
                  # 1       BacterialLeaf/2.jpg      0
                  # 2       Blast/1.jpg              1
                  # 3       Blast/2.jpg              1
                  # 4       Blast/3.jpg              1
                  # 5       BrownSpot/1.jpg          2
                  # 6       Healthy/1.jpg            3
                  # 7       Healthy/2.jpg            3
                  # 8       Tungro/1.jpg             7
                  # 9       grassy/1.jpg             8
                 # 10      yellow/1.jpg             10
                  # ...

    def __len__(self):
        return len(self.samples) #__len__ tells PyTorch the total number of images in your dataset.

    def __getitem__(self, idx):
        path, label = self.samples[idx] #This line fetches the image path and its class label for the requested index. So we don't call the function only DataLoader will call that function.

        # Read image
        image = cv2.imread(path) #It opens the image file stored at path and converts it into a matrix of pixel values. [[[12, 140, 220], [34, 90, 10]],[[255, 180, 70], [0, 40, 200]]] Each pixel = 3 numbers (Blue, Green, Red)

        if image is None:
            # corrupted image protection
            image = np.zeros((224,224,3), dtype=np.uint8) #Sometimes in real datasets:image file damaged, wrong path, empty file, unsupported format. If we continue → program crashes ❌ Instead of stopping training, we replace the broken image with a blank image. dtype = data type of each pixel value, np.uint8,Unsigned Integer 8-bit 0-255. Create a grid of pixels where every color value is a whole number between 0 and 255

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) #this fixes the color order of the image. OpenCV reads colors in:BGR(Blue,Green,Red). But deep learning libraries (PyTorch, PIL, matplotlib) expect: RGB(Red,Green,Blue), It doesn’t change the image content — only channel order. Pixel from OpenCV:BGR After conversion: RGB Now correct colors. It converts OpenCV’s BGR image into standard RGB so the CNN sees correct colors.

        if self.transform:
            image = self.transform(image=image)["image"] #(image (parameter name)) = (opencv image matrix), After this runs:result = self.transform(image=image) Albumentations returns a dictionary, not the image directly. Like this:"image": processed_image_tensor}So the processed image is stored inside the key "image". Python allows direct extraction from the returned value.Step-by-step way: result = self.transform(image=image) image = result["image"]. Short way (what your code uses):image = self.transform(image=image)["image"]

        return image, torch.tensor(label, dtype=torch.long) #It changes the label from a normal number into a PyTorch number the model can understand. Before: label = 3 ← normal Python integer. After: tensor(3)        ← PyTorch tensor. Why torch.long: Because classification loss (CrossEntropyLoss) wants the answer as a whole number category, not decimal. 3.0 (float) ❌ 3 (long integer) ✔


In [ ]:
train_dataset = RiceDiseaseDataset(
    root_dir=DATASET_PATH,
    classes=classes,
    transform=train_transform
)

val_dataset = RiceDiseaseDataset(
    root_dir=DATASET_PATH,
    classes=classes,
    transform=val_transform
)


In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(train_dataset))
val_size   = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size]) #PyTorch does NOT give a normal dataset. It gives a Subset object.

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 13281
Val samples: 3321


In [ ]:
print(type(train_dataset)) #Subset object.

<class 'torch.utils.data.dataset.Subset'>


In [ ]:
print(dir(train_dataset)) # indise the Subset object we have dataset (Original full dataset) & indices (list of selected image positions)

['__add__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getitems__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'dataset', 'indices']


In [ ]:
#this code is used to collect all labels from the training split.
train_labels = []

for idx in train_dataset.indices:
    _, label = train_dataset.dataset.samples[idx] #_, label: here we are ignoring the path and accept the label,
    train_labels.append(label) #Example: train_labels = [2, 2, 2, 1, 0, 0, 0]

#     | Thing     | Stores                       |
# | --------- | ---------------------------- |
# | `indices` | which images belong to train |
# | `samples` | path + class label           |
# | `label`   | which class (0–10)           |

# 👉 samples does NOT belong to the Subset
# 👉 samples belongs to your original dataset
# train_dataset (Subset)
#     ├── indices → [5, 22, 104, 87, 2 ...]
#     └── dataset → RiceDiseaseDataset
#                      └── samples → (path, label)



In [ ]:
from collections import Counter
label_count = Counter(train_labels)
print(label_count)


Counter({1: 2988, 6: 1834, 2: 1751, 4: 1609, 0: 1428, 3: 1403, 7: 1165, 5: 823, 9: 119, 8: 117, 10: 44})


In [ ]:

# It is used to handle class imbalance
# (when some diseases have many images and some have very few).
import torch

class_sample_count = torch.tensor([label_count[i] for i in range(len(classes))])
class_weights = 1. / class_sample_count
print(class_weights)

# 50 images → 1/50 = 0.02  🔥 big importance
# 3765 images → 1/3765 ≈ 0.00026  low importance
# class_weights

# Weight per class (rare disease → bigger weight)

# Example:

# | Class | Images | Weight   |
# | ----- | ------ | -------- |
# | 0     | 1472   | 0.0006   |
# | 1     | 3034   | 0.0003   |
# | 8     | 125    | 0.008    |
# | 10    | 40     | 0.025 🔥 |


tensor([0.0007, 0.0003, 0.0006, 0.0007, 0.0006, 0.0012, 0.0005, 0.0009, 0.0085,
        0.0084, 0.0227])


In [ ]:
#this line converts class weights → weight for every single image.

samples_weight = torch.tensor([class_weights[label] for label in train_labels]) #torch.tensor() converts a single list into a single 1D tensor
print(samples_weight[:20])



tensor([0.0007, 0.0007, 0.0003, 0.0012, 0.0009, 0.0085, 0.0005, 0.0005, 0.0012,
        0.0003, 0.0007, 0.0007, 0.0007, 0.0003, 0.0005, 0.0003, 0.0007, 0.0006,
        0.0006, 0.0003])


In [ ]:
from torch.utils.data import WeightedRandomSampler

sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels[:10])


torch.Size([32, 3, 224, 224])
tensor([9, 4, 6, 4, 0, 4, 2, 2, 3, 3])


In [ ]:
import torch.nn as nn
import torchvision.models as models

NUM_CLASSES = 11

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.cuda()
print(model.fc)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


Linear(in_features=512, out_features=11, bias=True)


In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()


In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)


In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.3,
    patience=2
)


In [ ]:
from tqdm import tqdm
import torch

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):
        images = images.cuda()
        labels = labels.cuda()

        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # accuracy
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    return total_loss / len(loader), acc


In [ ]:
def validate(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.cuda()
            labels = labels.cuda()

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    return total_loss / len(loader), acc


In [ ]:
EPOCHS = 12

best_val_loss = float('inf')

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = validate(model, val_loader, criterion)

    # scheduler step
    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rice_model.pth")
        print("✅ Best model saved")



Epoch 1/12


100%|██████████| 416/416 [24:23<00:00,  3.52s/it]


Train Loss: 0.6452 | Train Acc: 0.7778
Val   Loss: 0.7807 | Val   Acc: 0.7425
✅ Best model saved

Epoch 2/12


100%|██████████| 416/416 [13:18<00:00,  1.92s/it]


Train Loss: 0.3818 | Train Acc: 0.8709
Val   Loss: 0.4744 | Val   Acc: 0.8398
✅ Best model saved

Epoch 3/12


...

Train Loss: 0.1456 | Train Acc: 0.9541
Val   Loss: 0.2071 | Val   Acc: 0.9289

Epoch 11/12


100%|██████████| 416/416 [02:29<00:00,  2.79it/s]


Train Loss: 0.1235 | Train Acc: 0.9637
Val   Loss: 0.2319 | Val   Acc: 0.9214

Epoch 12/12


100%|██████████| 416/416 [02:30<00:00,  2.77it/s]


Train Loss: 0.1263 | Train Acc: 0.9582
Val   Loss: 0.1644 | Val   Acc: 0.9407
✅ Best model saved


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/DDM.pth")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(
    torch.load("/content/drive/MyDrive/DDM.pth",
               map_location=device)
)

model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

NUM_CLASSES = 11

# rebuild architecture
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# ✅ define device FIRST
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ move model to device
model = model.to(device)

# ✅ load saved weights
model.load_state_dict(
    torch.load("/content/drive/MyDrive/DDM.pth",
               map_location=device)
)

# ✅ set to evaluation mode
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [ ]:
import os

print("Current directory:", os.getcwd())
print("Files in /content:", os.listdir("/content"))
print("Files in Drive:", os.listdir("/content/drive/MyDrive"))

Current directory: /content
Files in /content: ['.config', 'best_rice_model.pth', 'drive', 'sample_data']
Files in Drive: ['Colab Notebooks', 'bgm', 'MS office', 'InternProjDoc', 'FreeZon', 'SHRI', 'WhatsApp Image 2025-08-12 at 20.56.26_b64ebc22.jpg', 'Documnets', 'DataSet', 'CS-Horizon-3rd-Edition-1_copy.pdf', 'Untitled document (2).gdoc', 'Backup', 'Untitled presentation.gslides', 'Untitled document (1).gdoc', 'Untitled document.gdoc', 'Aegis-main', 'blockhdd.py', 'codeinjection.py', 'detectexe.py', 'Aegis.py', 'SHRI031.gslides', 'American College Staff and HODs.gsheet', 'ProjectImages', 'GreenLensAI', 'best_rice_model.pth', 'DDM.pth']


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/DDM.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn
from torchvision import transforms
from PIL import Image

NUM_CLASSES = 11

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model.load_state_dict(torch.load("/content/drive/MyDrive/DDM.pth", map_location="cpu"))
model.eval()

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

img = Image.open("/content/drive/MyDrive/DataSet/GreenLens/RCLD/Narrow Brown Leaf Spot/IMG_20231014_173039.jpg").convert("RGB")
img = transform(img).unsqueeze(0)

with torch.no_grad():
    out = model(img)
    probs = torch.softmax(out, dim=1)

print(probs)

tensor([[0.1327, 0.0388, 0.0388, 0.0125, 0.0351, 0.0472, 0.0402, 0.0250, 0.1244,
         0.4118, 0.0934]])
